In [2]:
!pip install scapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.2 MB/s eta 0:00:00


In [ ]:
# --- Gamma-Prime (γ') Component Extraction Script (v3 - Directional) ---
# This script extracts burst statistics SEPARATELY for
# Client-to-Server (C2S) and Server-to-Client (S2C) directions.
#
# UPDATES:
# 1. Separated burst logic for C2S and S2C.
# 2. Added 'burst_density' (Bytes per Second) as a feature.

print("--- Initializing Gamma-Prime (γ') v3 (Directional) Script ---")

try:
    import scapy.all as scapy
except ImportError:
    print("Please run '!pip install scapy' first.")

import os
import collections
import time
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scapy.all import rdpcap, IP

# --- CONFIGURATION ---
FLOW_DIR = "/content/drive/MyDrive/1 Skripsi/Notebook/VPNOnlyDataset"
OUTPUT_CSV = "/content/drive/MyDrive/1 Skripsi/VPNOnly-gamma_prime_component_v3.csv"
BURST_IDLE_THRESHOLD = 1.0

# --- LABELING MAP ---
KEYWORD_MAP = collections.OrderedDict([
    ('facebook_chat', ('Facebook', 'Chat')),
    ('facebookchat', ('Facebook', 'Chat')),
    ('hangouts_chat', ('Hangout', 'Chat')),
    ('hangout_chat', ('Hangout', 'Chat')),
    ('gmailchat', ('Gmail', 'Chat')),
    ('icq_chat', ('ICQ', 'Chat')),
    ('icqchat', ('ICQ', 'Chat')),
    ('skype_chat', ('Skype', 'Chat')),
    ('aim_chat', ('AIM Chat', 'Chat')),
    ('aimchat', ('AIM Chat', 'Chat')),
    ('facebook_audio', ('Facebook', 'VoIP')),
    ('hangouts_audio', ('Hangout', 'VoIP')),
    ('skype_audio', ('Skype', 'VoIP')),
    ('voipbuster', ('VOIPBuster', 'VoIP')),
    ('facebook_video', ('Facebook', 'VoIP')),
    ('hangouts_video', ('Hangout', 'VoIP')),
    ('skype_video', ('Skype', 'VoIP')),
    ('skype_file', ('Skype', 'File Transfer')),
    ('ftps', ('FTP', 'File Transfer')),
    ('sftp', ('SFTP', 'File Transfer')),
    ('scp', ('SCP', 'File Transfer')),
    ('ftp', ('FTP', 'File Transfer')),
    ('email', ('Email', 'Email')),
    ('gmail', ('Gmail', 'Email')),
    ('netflix', ('Netflix', 'Streaming')),
    ('spotify', ('Spotify', 'Streaming')),
    ('vimeo', ('Vimeo', 'Streaming')),
    ('youtube', ('YouTube', 'Streaming')),
    ('bittorrent', ('BitTorrent', 'P2P')),
])

TARGET_APPS = {'Skype', 'Email', 'SCP', 'VOIPBuster', 'YouTube', 'BitTorrent'}

def get_flow_labels(filename):
    lower_filename = filename.lower()
    binary_type = 'VPN' if lower_filename.startswith('vpn_') else 'NonVPN'
    for keyword, (application, category) in KEYWORD_MAP.items():
        if keyword in lower_filename:
            if application not in TARGET_APPS:
                if application == 'SCP': return 'SCP', 'File Transfer', binary_type
            return application, category, binary_type

    # Fallback
    if 'scp' in lower_filename: return 'SCP', 'File Transfer', binary_type
    if 'email' in lower_filename: return 'Email', 'Email', binary_type
    if 'youtube' in lower_filename: return 'YouTube', 'Streaming', binary_type
    if 'bittorrent' in lower_filename: return 'BitTorrent', 'P2P', binary_type
    if 'skype' in lower_filename: return 'Skype', 'Unknown', binary_type
    if 'voipbuster' in lower_filename: return 'VOIPBuster', 'VoIP', binary_type
    return None, None, None

def calculate_stats(data_list, prefix):
    stats = {}
    stat_names = ['count', 'sum', 'mean', 'std', 'min', 'max', 'median', 'p25', 'p75']
    for name in stat_names:
        stats[f"{prefix}_{name}"] = 0.0

    if not data_list:
        return stats

    arr = np.array(data_list)
    stats[f"{prefix}_count"] = float(arr.size)
    stats[f"{prefix}_sum"] = float(np.sum(arr))
    stats[f"{prefix}_mean"] = float(np.mean(arr))
    stats[f"{prefix}_min"] = float(np.min(arr))
    stats[f"{prefix}_max"] = float(np.max(arr))
    stats[f"{prefix}_median"] = float(np.median(arr))
    stats[f"{prefix}_p25"] = float(np.percentile(arr, 25))
    stats[f"{prefix}_p75"] = float(np.percentile(arr, 75))

    if arr.size > 1:
        stats[f"{prefix}_std"] = float(np.std(arr))

    return stats

def get_burst_features(packet_list, prefix):
    """
    Helper function to calculate burst stats for a specific list of packets.
    packet_list: list of (time, size) tuples
    """
    if not packet_list:
        # Return empty stats with correct keys
        empty_feats = {}
        empty_feats[f"{prefix}_burst_count"] = 0.0
        empty_feats.update(calculate_stats([], f"{prefix}_burst_pkt_count"))
        empty_feats.update(calculate_stats([], f"{prefix}_burst_vol"))
        empty_feats.update(calculate_stats([], f"{prefix}_burst_dur"))
        empty_feats.update(calculate_stats([], f"{prefix}_burst_idle"))
        return empty_feats

    # Sort by time
    packet_list.sort(key=lambda x: x[0])

    burst_packet_counts = []
    burst_volumes = []
    burst_durations = []
    burst_idle_times = []

    # Init first burst
    current_burst_packets = 1
    current_burst_volume = packet_list[0][1]
    current_burst_start_time = packet_list[0][0]
    last_packet_time = packet_list[0][0]

    for (pkt_time, pkt_size) in packet_list[1:]:
        idle_time = pkt_time - last_packet_time

        if idle_time < BURST_IDLE_THRESHOLD:
            current_burst_packets += 1
            current_burst_volume += pkt_size
        else:
            # End current burst
            burst_duration = last_packet_time - current_burst_start_time
            burst_packet_counts.append(current_burst_packets)
            burst_volumes.append(current_burst_volume)
            burst_durations.append(burst_duration)
            burst_idle_times.append(idle_time)

            # Start new burst
            current_burst_packets = 1
            current_burst_volume = pkt_size
            current_burst_start_time = pkt_time

        last_packet_time = pkt_time

    # Final burst
    burst_duration = last_packet_time - current_burst_start_time
    burst_packet_counts.append(current_burst_packets)
    burst_volumes.append(current_burst_volume)
    burst_durations.append(burst_duration)

    # Compile Features
    features = {}
    features[f"{prefix}_total_bursts"] = float(len(burst_packet_counts))

    features.update(calculate_stats(burst_packet_counts, f"{prefix}_burst_pkts"))
    features.update(calculate_stats(burst_volumes, f"{prefix}_burst_vol"))
    features.update(calculate_stats(burst_durations, f"{prefix}_burst_dur"))
    features.update(calculate_stats(burst_idle_times, f"{prefix}_burst_idle"))

    return features

def process_pcap_file(filename, base_dir):
    filepath = os.path.join(base_dir, filename)
    application, category, binary_type = get_flow_labels(filename)
    if application is None: return None

    try:
        packets = rdpcap(filepath)

        # Identify Client IP
        client_ip = None
        for pkt in packets:
            if IP in pkt:
                client_ip = pkt[IP].src
                break
        if client_ip is None: return None

        c2s_packets = []
        s2c_packets = []

        for pkt in packets:
            if IP in pkt:
                packet_size = float(pkt[IP].len)
                packet_time = float(pkt.time)

                if pkt[IP].src == client_ip:
                    c2s_packets.append((packet_time, packet_size))
                elif pkt[IP].dst == client_ip:
                    s2c_packets.append((packet_time, packet_size))

    except Exception:
        return None

    if not c2s_packets and not s2c_packets: return None

    # Calculate Directional Features
    features = {}
    features.update(get_burst_features(c2s_packets, "c2s"))
    features.update(get_burst_features(s2c_packets, "s2c"))

    # Add Labels
    features['filename'] = filename
    features['application'] = application
    features['category'] = category
    features['binary_type'] = binary_type

    return features

def main():
    print(f"Reading from: {FLOW_DIR}")
    if not os.path.isdir(FLOW_DIR):
        print("Source directory not found.")
        return

    filenames = [f for f in os.listdir(FLOW_DIR) if f.endswith('.pcap')]
    print(f"Found {len(filenames)} files. Processing...")

    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(process_pcap_file)(f, FLOW_DIR) for f in filenames
    )

    valid_results = [r for r in results if r is not None]

    if valid_results:
        df = pd.DataFrame(valid_results)
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"Saved Directional Gamma features to {OUTPUT_CSV}")
    else:
        print("No valid results.")

if __name__ == "__main__":
    main()

--- Initializing Gamma-Prime (γ') v2 Component Script ---
All libraries imported successfully.

--- PART 1: Extracting Gamma-Prime (γ') Features ---
Reading from: /content/drive/MyDrive/1 Skripsi/Notebook/VPNOnlyDataset
Using Burst Idle Threshold: 1.0s
Found 2730 .pcap files in the directory.
Processing files in parallel... (This may take several minutes)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    7.6s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   15.5s
[Parallel(n_jobs=-1)]: Done 288 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 936 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 1334 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 1942 tasks      | elapsed:  2.7min
/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done 2447 tasks      | elapsed:  4.1min
[Parallel(n_jobs=-1)]: Done 2730 out of 2730 | elapsed:  6.5min finished


File processing finished in 388.19 seconds.
Successfully processed 2730 files.
Skipped 0 empty/corrupted/unlabeled files.

--- PART 2: Saving Final Dataset ---
Successfully saved final gamma-prime component (v2) to:
/content/drive/MyDrive/1 Skripsi/VPNOnly-gamma_prime_component_v2.csv

--- Gamma-Prime (γ') v2 Script Finished ---
